# 🔧 題目 1：零售 POS 銷售分析
# Mini Data Pipeline 工作坊

> **情境**：你是一家零售連鎖集團的資料顧問。老闆想知道哪些商品最暢銷、哪些客戶最有價值、各國市場表現如何。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`
>
> **資料**：[Kaggle: Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`

---

### 📋 今日目標

| 必做 | 選做 |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | ⭐ FastAPI API |
| ✅ SQL 查詢統計分析 | ⭐ Streamlit Dashboard |
| ✅ LLM 品類分類 | |
| ✅ 顧問報告 output/report.md | |

### 🗺️ 這份 Notebook 的標記說明

| 標記 | 意思 |
|------|------|
| `# TODO:` | **你要寫程式碼的地方** |
| `# 💡 提示:` | 告訴你可以用什麼方法 |
| `# ✅ 預期結果:` | 跑完應該看到什麼 |
| `（不需要改）` | 直接跑就好 |


## Section 0：環境設定

直接跑下面兩格，不需要改。


In [ ]:
# （不需要改）安裝與載入套件
import pandas as pd
import sqlite3
import os
import json

print("✅ 套件載入完成")


In [ ]:
# （不需要改）API Key 設定
OPENAI_API_KEY = ""  # 講師會提供共用 Key

if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]

print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback（不影響完成度）")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> **目標**：把 CSV 讀進來，寫入 SQLite 的 `raw_orders` 表。
> 這是 pipeline 第一步：**資料進入系統**。
>
> 為什麼寫入資料庫？
> - 真實 DE 工作中，資料不會一直是 CSV
> - 寫入資料庫後，可以用 SQL 查詢、可以被 API 存取
> - 今天用 SQLite（零安裝），後續升級到 MySQL / BigQuery 概念一樣


### Step 1-1：讀取 CSV

在下面的 cell 完成以下步驟：
1. 用 `pd.read_csv()` 讀取 `data/raw/topic_1/orders.csv`，存成 `df_raw`
2. 印出筆數和欄位名稱
3. 用 `.head()` 看前幾筆

> 💡 提示：`pd.read_csv("檔案路徑")` 回傳一個 DataFrame
> ✅ 預期結果：2,000 筆、9 個欄位（invoice_id, stock_code, description, quantity, ...）


In [ ]:
# TODO:
# 1. df_raw = pd.read_csv(???)
# 2. print 筆數和欄位
# 3. df_raw.head()


### Step 1-2：檢查資料品質

在下面的 cell 完成以下步驟：
1. 用 `df_raw.dtypes` 看每個欄位的型別
2. 用 `df_raw.isnull().sum()` 看哪些欄位有缺漏值
3. 用 `df_raw.describe()` 看數值欄位的統計（最小、最大、平均）

> 💡 提示：這三個是資料探索的標準動作，每次拿到新資料都先做
> ✅ 預期結果：quantity 和 unit_price 是數值、invoice_date 是字串（之後要轉）


In [ ]:
# TODO:
# 1. print(df_raw.dtypes)
# 2. print(df_raw.isnull().sum())
# 3. print(df_raw.describe())


### Step 1-3：建立 SQLite 資料庫 + 寫入 raw 表

在下面的 cell 完成以下步驟：
1. 用 `sqlite3.connect("pipeline.db")` 建立資料庫連線，存成 `conn`
2. 用 `df_raw.to_sql("raw_orders", conn, if_exists="replace", index=False)` 寫入
3. 用 `pd.read_sql("SELECT COUNT(*) as total FROM raw_orders", conn)` 驗證

> 💡 提示：
> - `sqlite3.connect()` 如果檔案不存在會自動建立
> - `to_sql()` 的 `if_exists="replace"` 表示如果表已存在就覆蓋
> - `index=False` 表示不把 DataFrame 的 index 存進去
> ✅ 預期結果：印出「raw_orders: 2000 筆」


In [ ]:
# TODO:
# 1. conn = sqlite3.connect(???)
# 2. df_raw.to_sql(???)
# 3. 用 pd.read_sql 驗證筆數


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> **目標**：從 `raw_orders` 讀出 → 清洗 → 寫入 `cleaned_orders`。
>
> 💡 注意：這裡從**資料庫**讀，不是從 CSV！這就是 pipeline 的概念。


### Step 2-1：從 raw 表讀出資料

在下面的 cell：
1. 用 `pd.read_sql("SELECT * FROM raw_orders", conn)` 讀出資料，存成 `df`
2. 記下清洗前的筆數（`before = len(df)`）

> 💡 提示：`pd.read_sql(SQL語句, 連線)` 可以用 SQL 從資料庫取資料
> ✅ 預期結果：df 有 2000 筆


In [ ]:
# TODO:
# 1. df = pd.read_sql(???)
# 2. before = len(df)
# 3. print


### Step 2-2：處理缺漏值

在下面的 cell：
1. 用 `df.dropna(subset=["description", "customer_id"])` 刪除這兩個欄位為空的列
2. 把結果存回 `df`（覆蓋原來的）
3. 印出清洗前後筆數比較

> 💡 提示：`dropna(subset=[...])` 只看指定欄位有沒有空值
> ✅ 預期結果：可能會少幾筆，也可能不變（取決於資料品質）


In [ ]:
# TODO:
# 1. df = df.dropna(subset=[???])
# 2. print(f"清洗前: {before} → 清洗後: {len(df)} 筆")


### Step 2-3：日期轉換 + 新增時間特徵欄位

在下面的 cell 依序完成：
1. 把 `invoice_date` 轉成 datetime 型別：`pd.to_datetime(df["invoice_date"])`
2. 從 datetime 提取 4 個新欄位：
   - `year`：年份 → 用 `.dt.year`
   - `month`：月份 → 用 `.dt.month`
   - `day_of_week`：星期幾 → 用 `.dt.day_name()`
   - `hour`：小時 → 用 `.dt.hour`
3. 印出前 3 筆確認

> 💡 提示：先轉 datetime 存回同一個欄位，再從該欄位提取
> ```python
> df["欄位"] = pd.to_datetime(df["欄位"])
> df["新欄位"] = df["欄位"].dt.year
> ```
> ✅ 預期結果：df 多了 year, month, day_of_week, hour 四個欄位


In [ ]:
# TODO:
# 1. df["invoice_date"] = pd.to_datetime(???)
# 2. df["year"] = ???
# 3. df["month"] = ???
# 4. df["day_of_week"] = ???
# 5. df["hour"] = ???
# 6. 印出前 3 筆確認


### Step 2-4：計算總金額 + 過濾異常值

在下面的 cell：
1. 確認 `total_amount` 欄位 = `quantity × unit_price`
2. 過濾掉不合理的資料：`quantity ≤ 0` 或 `unit_price ≤ 0` 的列

> 💡 提示：
> ```python
> df["total_amount"] = df["quantity"] * df["unit_price"]
> df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]
> ```
> ✅ 預期結果：total_amount 都是正數


In [ ]:
# TODO:
# 1. df["total_amount"] = ???
# 2. df = df[(???) & (???)]
# 3. print(f"最終: {len(df)} 筆")


### 🏁 清洗檢查點

> 直接跑下面這格。全部 ✅ 才往下。如果有 ❌ 回去看上面哪步出錯。
>
> 這格做什麼：用 `assert` 語法自動檢查：
> - 沒有缺漏值
> - quantity 和 unit_price 都是正數
> - year, month 等新欄位存在


In [ ]:
# （不需要改）自動檢查
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值！回去看 Step 2-2"
assert (df["quantity"] > 0).all(), "❌ quantity 有非正值！回去看 Step 2-4"
assert (df["unit_price"] > 0).all(), "❌ unit_price 有非正值！回去看 Step 2-4"
assert "year" in df.columns, "❌ 缺少 year 欄位！回去看 Step 2-3"
assert "month" in df.columns, "❌ 缺少 month 欄位！回去看 Step 2-3"
assert "day_of_week" in df.columns, "❌ 缺少 day_of_week 欄位！回去看 Step 2-3"
assert "hour" in df.columns, "❌ 缺少 hour 欄位！回去看 Step 2-3"
print("✅ 全部檢查通過！")
print(f"   清洗後: {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned 表

在下面的 cell：
1. 用 `df.to_sql("cleaned_orders", conn, if_exists="replace", index=False)` 寫入
2. 用 SQL 驗證兩張表的筆數

> 💡 提示：跟 Step 1-3 一樣的寫法，只是表名不同
> ✅ 預期結果：cleaned_orders 筆數 ≤ raw_orders 筆數（清洗掉了髒資料）


In [ ]:
# TODO:
# 1. df.to_sql(???)
# 2. 驗證：印出 raw_orders 和 cleaned_orders 的筆數


---
## Section 3：SQL 統計分析

> **目標**：用 SQL 從 `cleaned_orders` 表做統計。
>
> 💡 為什麼用 SQL 不用 pandas？
> 真實工作中，資料在資料庫裡，分析師是寫 SQL 查詢，不是下載 CSV 再用 pandas。
> 今天兩個都練，但重點是體驗「**從資料庫查詢**」。
>
> 要回答的問題：
> 1. 哪些商品銷售額最高？
> 2. 哪些國家貢獻最多營收？
> 3. （自由發揮）你還想知道什麼？


### Step 3-1：商品銷售排行（範例 SQL，直接跑）

> 這格是範例，讓你看 `pd.read_sql()` + SQL 怎麼用。
> 觀察 SQL 語法：`SELECT` 選欄位、`GROUP BY` 分組、`ORDER BY` 排序、`LIMIT` 取前 N 筆。


In [ ]:
# （範例，直接跑）商品銷售排行
product_stats = pd.read_sql("""
SELECT description, 
       COUNT(*) as order_count,
       SUM(quantity) as total_qty,
       ROUND(SUM(total_amount), 2) as total_revenue
FROM cleaned_orders
GROUP BY description
ORDER BY total_revenue DESC
LIMIT 20
""", conn)
print("📊 商品銷售 Top 20：")
product_stats


### Step 3-2：各國銷售統計（你來寫 SQL）

在下面的 cell 寫一個 SQL 查詢，回答：**各國有多少客戶、多少訂單、多少營收？**

> 💡 提示（一步一步）：
> 1. `SELECT country, ...` — 選國家和你要的統計
> 2. `COUNT(DISTINCT customer_id)` — 計算不重複的客戶數
> 3. `COUNT(*)` — 計算訂單數
> 4. `ROUND(SUM(total_amount), 2)` — 計算營收（取小數 2 位）
> 5. `FROM cleaned_orders` — 從 cleaned 表查
> 6. `GROUP BY country` — 按國家分組
> 7. `ORDER BY revenue DESC` — 按營收排序
> ✅ 預期結果：United Kingdom 的營收最高


In [ ]:
# TODO: 寫你的 SQL
country_stats = pd.read_sql("""
SELECT 
    ???
FROM cleaned_orders
GROUP BY ???
ORDER BY ???
""", conn)
country_stats


### Step 3-3：自由發揮 — 你還想知道什麼？

用 SQL 查你好奇的東西。以下是一些靈感：

| 想知道什麼 | SQL 提示 |
|-----------|---------|
| 每月銷售趨勢 | `GROUP BY year, month` |
| 尖峰購買時段 | `GROUP BY hour` |
| 客戶消費排行 | `GROUP BY customer_id` + `SUM(total_amount)` |
| 哪個星期幾賣最多 | `GROUP BY day_of_week` |


In [ ]:
# TODO: 自由發揮！寫你自己的 SQL 查詢
my_analysis = pd.read_sql("""

""", conn)
my_analysis


### Step 3-4：存統計結果

在下面的 cell：
1. 建立 `data/processed/` 資料夾
2. 把 product_stats 存成 CSV

> 💡 提示：
> ```python
> os.makedirs("data/processed", exist_ok=True)
> product_stats.to_csv("data/processed/product_stats.csv", index=False)
> ```


In [ ]:
# TODO:
# 1. os.makedirs(???)
# 2. product_stats.to_csv(???)
# 3. 如果你有 country_stats，也存一份


---
## Section 4：LLM 加值分析

> **目標**：用 LLM 對商品描述做品類分類，結果寫入 `analyzed_orders` 表。
>
> 下面的 helper 函式**已寫好**（因為 API 呼叫比較複雜，不適合從零寫）。
> **你要做的是**：
> 1. 跑單筆測試，看分類結果合不合理
> 2. 如果不滿意，改 `_llm_fallback` 裡的**關鍵字規則**
> 3. 呼叫函式做批次分析
> 4. 把結果寫入 analyzed 表


In [ ]:
# （不需要改，除非你想調整 prompt 或 fallback 規則）
import requests

def llm_analyze(text, api_key=None):
    if api_key:
        return _llm_api(text, api_key)
    return _llm_fallback(text)

def _llm_api(text, api_key):
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}

商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3},
            timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except:
        return _llm_fallback(text)

def _llm_fallback(text):
    # 🎯 你可以加更多關鍵字來改善分類準確度！
    t = text.lower()
    if any(w in t for w in ["christmas", "xmas", "santa", "winter"]):
        cat = "季節商品"
    elif any(w in t for w in ["candle", "holder", "frame", "lamp"]):
        cat = "家飾"
    elif any(w in t for w in ["cup", "mug", "plate", "bowl"]):
        cat = "餐具"
    elif any(w in t for w in ["pen", "pencil", "notebook", "card"]):
        cat = "文具"
    elif any(w in t for w in ["gift", "bag", "box", "ribbon"]):
        cat = "禮品"
    else:
        cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}

print("✅ LLM Helper 已定義（有 API Key 用 API，沒有用 fallback 規則版）")


### Step 4-1：單筆測試

在下面的 cell：
1. 從 df 取第一筆的 description
2. 呼叫 `llm_analyze(文字, API_KEY)` 分析
3. 印出分類結果，看合不合理

> 💡 提示：
> ```python
> test_text = df["description"].iloc[0]
> result = llm_analyze(test_text, OPENAI_API_KEY if OPENAI_API_KEY else None)
> ```
> ✅ 預期結果：回傳 dict，有 category 和 insight 兩個 key


In [ ]:
# TODO:
# 1. test_text = df["description"].iloc[???]
# 2. result = llm_analyze(???)
# 3. print 分類結果
# 💡 不滿意結果？回去改 _llm_fallback 的關鍵字規則！


### Step 4-2：批次分析

在下面的 cell：
1. 設定 `BATCH_SIZE`（先用 50，確認品質後可以改大）
2. 用 for 迴圈對 df 前 BATCH_SIZE 筆的 description 呼叫 `llm_analyze()`
3. 把每筆結果 append 到 results 列表
4. 每 10 筆印一次進度

> 💡 提示：
> ```python
> results = []
> for i, row in df.head(BATCH_SIZE).iterrows():
>     r = llm_analyze(str(row["description"]), api_key)
>     results.append(r)
> ```
> ⏱ fallback 版幾秒完成；API 版 50 筆約 2-3 分鐘


In [ ]:
# TODO:
# 1. BATCH_SIZE = 50
# 2. api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
# 3. 用 for 迴圈跑批次分析
# 4. 每 10 筆印進度


### Step 4-3：整理結果 + 寫入 analyzed 表

在下面的 cell：
1. 把 df 前 BATCH_SIZE 筆複製一份 → `df_analyzed`
2. 把 results 裡的 category 和 insight 加成新欄位
3. 用 `to_sql("analyzed_orders", ...)` 寫入 SQLite
4. 印出三表狀態確認

> 💡 提示：
> ```python
> df_analyzed = df.head(BATCH_SIZE).copy()
> df_analyzed["category"] = [r["category"] for r in results]
> df_analyzed["llm_insight"] = [r["insight"] for r in results]
> df_analyzed.to_sql("analyzed_orders", conn, if_exists="replace", index=False)
> ```
> ✅ 預期結果：三表都有資料（raw > cleaned ≥ analyzed）


In [ ]:
# TODO:
# 1. df_analyzed = df.head(???).copy()
# 2. df_analyzed["category"] = ???
# 3. df_analyzed["llm_insight"] = ???
# 4. df_analyzed.to_sql(???)
# 5. 印出三表各自的筆數


---
## Section 5：驗證 pipeline

> 直接跑下面這格。這是確認三張表的資料一致的 **data lineage** 驗證。
>
> 這格做什麼：用一條 SQL 同時查三表的筆數，確認 raw → cleaned → analyzed 的資料流。


In [ ]:
# （不需要改）跨表查詢驗證
lineage = pd.read_sql("""
SELECT 'raw_orders' as layer, COUNT(*) as rows FROM raw_orders
UNION ALL SELECT 'cleaned_orders', COUNT(*) FROM cleaned_orders
UNION ALL SELECT 'analyzed_orders', COUNT(*) FROM analyzed_orders
""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))
print()
print("💡 raw → cleaned 減少 = 清洗掉了髒資料")
print("💡 cleaned → analyzed 減少 = 只分析了前 N 筆（可以改 BATCH_SIZE 跑更多）")


---
## Section 6：產出報告

在下面的 cell 產出顧問報告。模板已給，**你要做的是**：
1. 用 SQL 查出報告需要的數字（總銷售額等）
2. 根據你的分析結果，在「建議」段落寫 2-3 條有數據支撐的建議
3. 存到 `output/report.md`

> 💡 提示：f-string 可以把變數嵌入字串 → `f"總銷售額：${total_rev:,.2f}"`


In [ ]:
# TODO:
# 1. 用 SQL 查出總銷售額
# total_rev = pd.read_sql("SELECT ROUND(SUM(total_amount),2) as r FROM cleaned_orders", conn)["r"][0]

# 2. 用你的分析結果填入報告
# cat_dist = df_analyzed["category"].value_counts().to_dict()

report = f"""# 零售 POS 銷售分析報告

## 資料概要
- 分析筆數：??? 筆交易
- 總銷售額：$???
- 資料來源：UCI Online Retail II

## 關鍵發現
（TODO: 根據 Section 3 的統計結果，寫 2-3 個發現）


## 品類分佈
（TODO: 根據 Section 4 的 LLM 分析結果，列出品類分佈）


## 建議
（TODO: 根據你的發現，寫 2-3 條有數據支撐的建議）


## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""

os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f:
    f.write(report)
print("✅ 報告已存到 output/report.md")


---
## Section 7：打包確認

> 直接跑下面這格。自動檢查所有產出是否齊全。
>
> 這格做什麼：逐一檢查 pipeline.db、data/processed/、output/ 裡的檔案和 SQLite 表是否存在。


In [ ]:
# （不需要改）自動檢查
checks = [
    ("pipeline.db", "SQLite 資料庫"),
    ("data/processed/product_stats.csv", "商品統計"),
    ("output/report.md", "顧問報告"),
]
print("📋 產出確認：")
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {desc}: {path}")
    if not exists: all_ok = False

if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ SQLite 表 {t}: {n} 筆")
        except:
            print(f"  ❌ SQLite 表 {t} 不存在")
            all_ok = False
    c.close()

print()
if all_ok:
    print("🎉 全部完成！你的 Mini Data Pipeline 已就緒。")
else:
    print("⚠️ 有缺漏，請回去補完。")
print()
print("📋 接下來：")
print("  1. 填寫 README（用講師提供的模板）")
print("  2. 填寫 docs/upgrade_plan.md")
print("  3. 準備 3 分鐘 Demo")
print("  4.（選做）繼續做 Section 8-10（FastAPI + Dashboard + 本地部署）")


---
## Section 8（選做）：FastAPI — 把分析結果變成 API

> **為什麼需要 API？**
> 你在前面的 Section 把分析結果存進了 SQLite 資料庫。但如果別人（前端工程師、PM、客戶）想看你的分析，
> 他們不可能打開你的 .db 檔案。
>
> **API 的角色**：把資料庫裡的資料「包裝」成網址，別人用瀏覽器或程式就能取得。
> 這就是真實 DE 的做法：**資料庫 → API → 前端**。
>
> **兩條路線（擇一）**：
> - 🅰️ **Notebook 內體驗**：在下面的 cell 直接寫一個簡易 API，Colab 就能跑
> - 🅱️ **本地完整版**：打開同資料夾的 `api.py`，用 `uvicorn` 啟動完整 FastAPI

### 路線 🅰️：在 Notebook 裡寫簡易 API

FastAPI 的核心概念很簡單：
1. 建立一個 `app` 物件
2. 用 `@app.get("/路徑")` 裝飾器定義 endpoint
3. endpoint 函式回傳 dict（自動轉 JSON）
4. 啟動 server

在下面的 cell，完成一個有 3 個 endpoint 的簡易 API：

> 💡 提示（step by step）：
> ```python
> # 1. 安裝 + import
> !pip install fastapi uvicorn nest_asyncio
> from fastapi import FastAPI
> import nest_asyncio
> nest_asyncio.apply()  # Colab 需要這行才能在 Notebook 裡跑 server
>
> # 2. 建立 app
> api = FastAPI(title="你的 API 名稱")
>
> # 3. 定義 endpoint — 用 @api.get("/路徑") 裝飾器
> @api.get("/health")
> def health():
>     return {"status": "ok"}
>
> # 4. 定義統計 endpoint — 從 SQLite 查詢回傳
> @api.get("/stats")
> def get_stats():
>     conn = sqlite3.connect("pipeline.db")
>     df = pd.read_sql("你的 SQL", conn)
>     conn.close()
>     return df.to_dict(orient="records")
> ```


---
## Section 9（選做）：Dashboard — 視覺化你的分析結果

> **為什麼需要 Dashboard？**
> 你的客戶（營運經理）不會看 CSV 或 SQL 查詢結果。
> 他們要的是「打開一個頁面，看到圖表和結論」。
>
> **兩條路線（擇一）**：
> - 🅰️ **Notebook 內體驗**：用 ipywidgets 做互動式 Dashboard（Colab 可跑）
> - 🅱️ **本地完整版**：打開同資料夾的 `app.py`，用 Streamlit 跑完整 Dashboard

### 路線 🅰️：在 Notebook 裡用 ipywidgets 做互動 Dashboard

ipywidgets 讓你在 Notebook cell 裡放下拉選單、滑桿等元件，
選了之後圖表自動更新——概念跟 Streamlit 一樣，只是跑在 Notebook 裡。

在下面的 cell，完成一個有「下拉選單 + 動態圖表」的互動 Dashboard：

> 💡 提示（step by step）：
> ```python
> import ipywidgets as widgets
> from IPython.display import display, clear_output
> import matplotlib.pyplot as plt
>
> # 1. 從 cleaned_orders 讀出資料
> df_dash = pd.read_sql("SELECT * FROM cleaned_orders", conn)
>
> # 2. 建立下拉選單
> country_dropdown = widgets.Dropdown(
>     options=["全部"] + sorted(df_dash["country"].unique().tolist()),
>     description="選國家："
> )
>
> # 3. 定義更新函式
> def update_dashboard(country):
>     clear_output(wait=True)
>     display(country_dropdown)  # 重新顯示下拉選單
>     data = df_dash if country == "全部" else df_dash[df_dash["country"] == country]
>     print(f"📊 {country}: {len(data)} 筆, 營收 ${data['total_amount'].sum():,.2f}")
>     data.groupby("description")["total_amount"].sum().sort_values().tail(10).plot.barh(figsize=(10,5))
>     plt.title(f"商品銷售額 Top 10 — {country}")
>     plt.tight_layout()
>     plt.show()
>
> # 4. 綁定互動
> widgets.interact(update_dashboard, country=country_dropdown)
> ```


---
## Section 10（選做）：本地部署 — 完整跑起來

> 這個 Section 不寫程式，而是給你**回家後的操作指引**。
> 工作坊當天先完成 Section 1-7，回家後照這裡把完整的 FastAPI + Streamlit 跑起來。

### 環境準備

```bash
# 1. 確認 Python 環境
python3 --version  # 需要 3.8+

# 2. 安裝套件
pip install fastapi uvicorn streamlit pandas

# 3. 進入你的題目資料夾
cd data/raw/topic_1
```

### 啟動 FastAPI

```bash
# 啟動 API server（會在 localhost:8000）
uvicorn api:app --reload --port 8000

# 開瀏覽器到 http://localhost:8000/docs
# 你會看到 Swagger UI，可以直接測試每個 endpoint
```

### 啟動 Streamlit

```bash
# 另開一個 terminal
streamlit run app.py

# 自動開啟瀏覽器，看到完整 Dashboard
```

### 完整架構

```
你的 Notebook      →  pipeline.db（SQLite 資料庫）
                         ↓
api.py（FastAPI）  →  把 DB 資料包裝成 API endpoint
                         ↓
app.py（Streamlit） →  打 API 或直讀 DB，呈現 Dashboard
```

### 後續升級（對應課程）

| 現在 | 升級後 | 對應課程 |
|------|--------|---------|
| SQLite | MySQL / BigQuery | 資料庫模組 |
| 手動跑 Notebook | Airflow DAG 自動排程 | Airflow 模組 |
| 本地跑 Streamlit | Docker 容器化 + 雲端部署 | Docker + GCP 模組 |
| 單人 pipeline | 團隊多人協作 | 期末團專 |

> 詳細升級計畫請填寫 [docs/upgrade_plan.md](../../docs/upgrade_plan.md)
